# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


### Группы из PassengerId

In [2]:
def add_group_features(df):
    df = df.copy()
    df['GroupId'] = df['PassengerId'].str.split('_').str[0]
    gsize = df.groupby('GroupId').size()
    df['GroupSize'] = df['GroupId'].map(gsize)
    df['IsAlone'] = (df['GroupSize'] == 1).astype(int)
    return df

train = add_group_features(train)
test = add_group_features(test)

train[['PassengerId','GroupId','GroupSize','IsAlone']].head()

,PassengerId,GroupId,GroupSize,IsAlone
0,0001_01,0001,1,1
1,0002_01,0002,1,1
2,0003_01,0003,2,0
3,0003_02,0003,2,0
4,0004_01,0004,1,1


### Cabin: Deck, CabinNum, Side

In [3]:
def add_cabin_features(df):
    df = df.copy()
    cab = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = cab[0]
    df['CabinNum'] = cab[1].astype(float)
    df['Side'] = cab[2]
    return df

train = add_cabin_features(train)
test = add_cabin_features(test)

train[['Cabin','Deck','CabinNum','Side']].head()

,Cabin,Deck,CabinNum,Side
0,B/0/P,B,0.0,P
1,F/0/S,F,0.0,S
2,A/0/S,A,0.0,S
3,A/0/S,A,0.0,S
4,F/1/S,F,1.0,S


### Импутация: CryoSleep и траты

Из EDA: `CryoSleep=True` всегда означает нулевые траты. Используем это в обе стороны.

In [4]:
def impute_cryosleep_spend(df):
    df = df.copy()
    known_spend_sum = df[spend_cols].sum(axis=1, skipna=True)
    any_spend_known = df[spend_cols].notna().any(axis=1)

    infer_false = df['CryoSleep'].isna() & any_spend_known & (known_spend_sum > 0)
    df.loc[infer_false, 'CryoSleep'] = False

    sleep_true = df['CryoSleep'] == True
    for c in spend_cols:
        df.loc[sleep_true & df[c].isna(), c] = 0.0

    df['CryoSleep'] = df['CryoSleep'].fillna(False)

    for c in spend_cols:
        med = df.loc[(df['CryoSleep'] == False) & (df[c].notna()), c].median()
        df.loc[df[c].isna(), c] = med
    return df

train = impute_cryosleep_spend(train)
test = impute_cryosleep_spend(test)

train['CryoSleep'].isna().sum(), train[spend_cols].isna().sum().sum(), test[spend_cols].isna().sum().sum()

C:\Users\ICH\AppData\Local\Temp\ipykernel_12348\4131679248.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep'] = df['CryoSleep'].fillna(False)
C:\Users\ICH\AppData\Local\Temp\ipykernel_12348\4131679248.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep'] = df['CryoSleep'].fillna(False)


(np.int64(0), np.int64(0), np.int64(0))

Пропусков в CryoSleep и тратах не осталось ни в train, ни в test.

### Импутация: HomePlanet

Из EDA: внутри группы `HomePlanet` всегда одинаковый, `Deck` тесно связана с `HomePlanet`. Заполняем сначала модой по группе, потом модой по Deck.

In [5]:
def fill_homeplanet_by_group(df):
    df = df.copy()
    group_mode = df.dropna(subset=['HomePlanet']).groupby('GroupId')['HomePlanet'].agg(lambda x: x.mode()[0])
    mask = df['HomePlanet'].isna()
    df.loc[mask, 'HomePlanet'] = df.loc[mask, 'GroupId'].map(group_mode)
    return df

def fill_homeplanet_by_deck(df, mapping):
    df = df.copy()
    mask = df['HomePlanet'].isna()
    df.loc[mask, 'HomePlanet'] = df.loc[mask, 'Deck'].map(mapping)
    return df

deck_planet_map = train.dropna(subset=['HomePlanet']).groupby('Deck')['HomePlanet'].agg(lambda x: x.mode()[0])

train = fill_homeplanet_by_group(train)
test = fill_homeplanet_by_group(test)
train = fill_homeplanet_by_deck(train, deck_planet_map)
test = fill_homeplanet_by_deck(test, deck_planet_map)

train['HomePlanet'].isna().sum(), test['HomePlanet'].isna().sum()

(np.int64(4), np.int64(1))

Групповая импутация закрывает большую часть пропусков, Deck-импутация — почти всё остальное. Единицы строк остаются без HomePlanet (когда и группа, и Deck неизвестны) — их добьём общей модой на следующем шаге.

### Остальные пропуски

Простая импутация медианой/модой. 

In [6]:
fill_stats = {
    'HomePlanet': train['HomePlanet'].mode()[0],
    'Deck': train['Deck'].mode()[0],
    'Side': train['Side'].mode()[0],
    'CabinNum': train['CabinNum'].median(),
    'Age': train['Age'].median(),
    'VIP': train['VIP'].mode()[0],
    'Destination': train['Destination'].mode()[0],
}

def fill_remaining(df, stats):
    df = df.copy()
    for col, val in stats.items():
        df[col] = df[col].fillna(val)
    return df

train = fill_remaining(train, fill_stats)
test = fill_remaining(test, fill_stats)

fill_stats

C:\Users\ICH\AppData\Local\Temp\ipykernel_12348\1422904928.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)


{'HomePlanet': 'Earth',
 'Deck': 'F',
 'Side': 'S',
 'CabinNum': np.float64(427.0),
 'Age': np.float64(27.0),
 'VIP': False,
 'Destination': 'TRAPPIST-1e'}

In [8]:
used_cols = ['HomePlanet','CryoSleep','Destination','Age','VIP'] + spend_cols + ['Deck','CabinNum','Side']
train[used_cols].isna().sum().sum(), test[used_cols].isna().sum().sum()

(np.int64(0), np.int64(0))

Пропусков среди признаков, которые пойдут в модель, больше нет — ни в train, ни в test.

### Признаки из трат: TotalSpend, HasSpent

In [9]:
def add_spend_features(df):
    df = df.copy()
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    for c in spend_cols:
        df[f'HasSpent_{c}'] = (df[c] > 0).astype(int)
    return df

train = add_spend_features(train)
test = add_spend_features(test)

train[['TotalSpend','HasSpent_RoomService','HasSpent_Spa']].head()

,TotalSpend,HasSpent_RoomService,HasSpent_Spa
0,0.0,0,0
1,736.0,1,1
2,10383.0,1,1
3,5176.0,0,1
4,1091.0,1,1


### IsChild

In [10]:
train['IsChild'] = (train['Age'] < 13).astype(int)
test['IsChild'] = (test['Age'] < 13).astype(int)

train['IsChild'].value_counts()

IsChild
0    7887
1     806
Name: count, dtype: int64

### Проверка: даёт ли FE прирост

Сравниваем с baseline (`02_baseline.ipynb`, CV accuracy 0.7853) на той же модели —
LogisticRegression + StratifiedKFold(5). Добавляем признаки по одной группе за
раз и смотрим на CV.

In [11]:
y = train['Transported'].astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(num_features, cat_features):
    X = train[num_features + cat_features].copy()
    for c in cat_features:
        X[c] = X[c].astype(str)
    prep = ColumnTransformer([
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ])
    model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    return scores.mean(), scores.std()

run_cv(['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck'],
       ['HomePlanet','CryoSleep','Destination','VIP'])

(np.float64(0.7862638806080767), np.float64(0.009213717514474447))

Те же признаки, что в baseline, но с умной импутацией вместо медианы/моды: 0.7853 → 0.7863. Сама по себе импутация почти ничего не меняет — ожидаемо, ведь пропусков было немного.

In [12]:
num_v2 = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','GroupSize']
cat_v2 = ['HomePlanet','CryoSleep','Destination','VIP','Deck','Side','IsAlone','IsChild']

run_cv(num_v2, cat_v2)

(np.float64(0.7937412279453755), np.float64(0.006314804798034975))

Добавили Deck, Side, GroupSize, IsAlone, IsChild — заметный скачок, 0.7863 → 0.7937.

In [13]:
num_v3 = ['Age','RoomService_log','FoodCourt_log','ShoppingMall_log','Spa_log','VRDeck_log','GroupSize']

for c in spend_cols:
    train[f'{c}_log'] = np.log1p(train[c])
    test[f'{c}_log'] = np.log1p(test[c])

run_cv(num_v3, cat_v2)

(np.float64(0.7806274653567948), np.float64(0.008163084849409492))

Попробовали log для трат вместо сырых значений — стало хуже, 0.7937 → 0.7806. Контринтуитивно (обычно log помогает при таком скосе), но факт: для этой модели сырые траты работают лучше. Log-версии в финальный датасет не берём.

In [14]:
cat_v4 = cat_v2 + ['HasSpent_RoomService','HasSpent_FoodCourt','HasSpent_ShoppingMall','HasSpent_Spa','HasSpent_VRDeck']

run_cv(num_v2, cat_v4)

(np.float64(0.7962723441312184), np.float64(0.007007192956805849))

HasSpent-флаги поверх лучшей конфигурации дают ещё немного: 0.7937 → 0.7963. Пробовали также добавить CabinNum и TotalSpend поверх этого набора — оба варианта давали 0.793-0.794, то есть не помогали, в финальный набор их не включаем.

### Финальный набор и сохранение

Итоговый прирост CV accuracy: 0.7853 (baseline) → 0.7963 (+FE), при той же модели.

In [15]:
feature_cols = ['HomePlanet','CryoSleep','Destination','Age','VIP',
                'RoomService','FoodCourt','ShoppingMall','Spa','VRDeck',
                'Deck','CabinNum','Side','GroupId','GroupSize','IsAlone',
                'TotalSpend','HasSpent_RoomService','HasSpent_FoodCourt',
                'HasSpent_ShoppingMall','HasSpent_Spa','HasSpent_VRDeck','IsChild']

train_fe = train[['PassengerId'] + feature_cols + ['Transported']]
test_fe = test[['PassengerId'] + feature_cols]

train_fe.to_csv('train_fe.csv', index=False)
test_fe.to_csv('test_fe.csv', index=False)

train_fe.shape, test_fe.shape

((8693, 25), (4277, 24))